# Head-to-head: BERTopic vs. Layer B, on the *same shelf*

The true apples-to-apples test. We build FoodScholar's **real Layer B** offline, pick one
populated `foods` shelf, then cluster **exactly those chunks** with BERTopic and compare the two
partitions directly.

**Fair-comparison choices:**
- **Same vectors.** Both cluster the cached **BGE-base** embeddings (Layer B's Pass-1 signal). We
  give BERTopic a *passthrough* dimensionality reduction so k-Means sees the raw vectors Leiden
  sees — isolating the *clustering method*, not the embedding or UMAP.
- **Swappable clusterer** (BERTopic's `hdbscan_model` slot takes any model). Default
  **k-Means with `n_clusters` = the shelf's Layer B theme count**: full coverage, equal K, so the
  agreement metrics measure *how* each method cuts the shelf — not how many clusters it makes.
- **No LLM / no services.** Layer B runs with `labeling.strategy="keyword"` and `alias_shelves=False`;
  embeddings are injected from cache, so nothing re-encodes.

What Layer B has that BERTopic doesn't: **Pass 2** (FoodOn entity co-occurrence). Where the two
partitions *disagree* is where that entity signal is doing work.

> The build cell takes **~4 min** (Layer A + attach + Layer B over 13k chunks). Companion notebook:
> `bertopic_stack_ablation.ipynb` (the bare-corpus / ontology-value view).

## 1 · Setup

In [1]:
try:
    import bertopic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "bertopic", "umap-learn", "plotly"], check=True)

import json, math, os
from collections import Counter
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE = ROOT / "data" / "cache"
SNAPSHOT = ROOT / "data" / "annotated.parquet"
RANDOM_STATE = 42

emb_files = sorted(CACHE.glob("bge_base_emb_*.npy"))
assert emb_files, "Run bertopic_explore.ipynb first to build data/cache/bge_base_emb_*.npy"
EMB_PATH = emb_files[-1]
IDS_PATH = CACHE / EMB_PATH.name.replace("emb", "ids").replace(".npy", ".json")
emb = np.load(EMB_PATH)
ids = json.loads(IDS_PATH.read_text())
id2vec = dict(zip(ids, emb))
print(f"cached vectors: {emb.shape}  ({EMB_PATH.name})")

# --- LLM setup for theme labels (Layer B §2 + BERTopic §4). Optional. --------------
# NOTE: a Jupyter kernel only inherits env vars that existed WHEN IT LAUNCHED. If you
# exported the key after starting Jupyter, the kernel can't see it — paste it below.
LLM_PROVIDER = "groq"   # "openrouter" | "anthropic" | "groq" | "openai" | "none"
LLM_API_KEY  = "***REMOVED-GROQ-KEY***"             # <-- paste a key here, OR leave "" to read it from the env
LLM_MODEL    = "llama-3.1-8b-instant"             # optional model override; "" = sensible default per provider
_ENV_VAR = {"openrouter": "OPENROUTER_API_KEY", "anthropic": "ANTHROPIC_API_KEY",
            "groq": "GROQ_API_KEY", "openai": "OPENAI_API_KEY"}.get(LLM_PROVIDER)
if LLM_API_KEY.strip() and _ENV_VAR:
    os.environ[_ENV_VAR] = LLM_API_KEY.strip()
LLM_ENABLED = bool(_ENV_VAR and os.environ.get(_ENV_VAR))
LLM_MODEL = LLM_MODEL or {"openrouter": "google/gemma-4-31b-it:free",
                          "anthropic": "claude-haiku-4-5-20251001",
                          "groq": "llama-3.3-70b-versatile",
                          "openai": "gpt-4o-mini"}.get(LLM_PROVIDER, "")
print(f"LLM: provider={LLM_PROVIDER} · key={'SET' if LLM_ENABLED else 'MISSING'} "
      f"({_ENV_VAR}) · model={LLM_MODEL or '-'}")

/mnt/miniconda3/envs/foodscholar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cached vectors: (13252, 768)  (bge_base_emb_13252.npy)
LLM: provider=groq · key=SET (GROQ_API_KEY) · model=llama-3.1-8b-instant


## 2 · Build the real Layer A + B (offline, ~4 min)

**Theme labels.** Clustering is deterministic, so the *comparison numbers are identical* whether labels
are c-TF-IDF keywords or LLM-written — the LLM only makes the **names** readable (`____ fruit ____` →
"Fruit washing & pesticide residue"). To enable, set `LLM_API_KEY` in §1 (or have the key in the env
*before* launching Jupyter). With a key the build labels every theme on the facet (~600 LLM calls), so
it runs longer; without one it falls back to deterministic keyword labels.

In [2]:
from foodscholar import FoodScholar
from foodscholar.config import FoodScholarConfig

# Use the provider/key/model resolved in §1. from_config does ${ENV} substitution, so the
# key set into os.environ above is what the provider client reads.
llm_cfg = {"primary": {"provider": LLM_PROVIDER, "model": LLM_MODEL}} if LLM_ENABLED else None
if not LLM_ENABLED:
    print("⚠ no LLM key — using deterministic keyword labels (set LLM_API_KEY in §1)")

cfg_dict = {
    "corpus": {"chunks_path": str(SNAPSHOT), "annotated_snapshot_path": str(SNAPSHOT)},
    "ontology": {"foodon_path": str(ROOT / "data" / "foodon.owl"),
                 "cache_path": str(ROOT / "data" / "foodon_cache.parquet")},
    "storage": {"chunk_store": {"backend": "memory"}, "graph_store": {"backend": "memory"}},
    "layer_a": {"projection": "backbone", "alias_shelves": bool(llm_cfg)},   # LLM aliases shelf names too
    "layer_b": {"pass1_mode": "per_shelf",
                "labeling": {"strategy": "llm" if llm_cfg else "keyword"}},
}
if llm_cfg:
    cfg_dict["llm"] = llm_cfg
cfg = FoodScholarConfig.model_validate(cfg_dict)

fs = FoodScholar.from_config(cfg)
fs.init()
fs.load_chunks(SNAPSHOT)
fs.chunk_store.update_embeddings_bulk([(i, id2vec[i].tolist(), "BAAI/bge-base-en-v1.5")
                                       for i in ids if i in id2vec])  # inject cached vectors
fs.attach_ontology(fs.load_ontology())
fs.build_entities()
fs.build_layer_a()
fs.attach()
art = fs.build_layer_b(facet="foods")
print(f"Layer B: {art.n_themes_total} themes across {art.n_shelves_themed} shelves "
      f"({art.n_themes_by_pass})  labels={'llm' if llm_cfg else 'keyword'}")

2026-06-05T11:39:22.803017Z [info     ] init.done                      chunk_store=memory config_hash=af1d3da6d1a3c76d entity_store=InMemoryEntityStore graph_store=memory
2026-06-05T11:39:28.197228Z [info     ] corpus.loaded                  config_hash=af1d3da6d1a3c76d n=13344
ontology.cache_hit path=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon_cache.parquet
2026-06-05T11:39:31.388918Z [info     ] ontology.loaded                cached=True n_terms=39278 source=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon.owl
2026-06-05T11:39:33.053578Z [info     ] build_entities.done            artifact_id=build_entities-2bbaec5c05a6 config_hash=af1d3da6d1a3c76d n_chunks_seen=13344 n_edges=44813 n_entities=3879
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https

Layer B: 599 themes across 90 shelves ({'global_similarity': 358, 'relatedness': 241})  labels=llm


## 3 · Pick a shelf

In [3]:
# Rank themed foods shelves by *directly-themed* chunk count (what the comparison actually runs on).
# `subtree_chunks` is the rollup over descendant shelves — bigger, but not what Layer B themed here.
def themed_chunks(themes):
    seen = set()
    for th in themes:
        seen.update(fs.graph_store.get_chunks_for_theme(th.theme_id))
    return seen

rows = [(s, t, themed_chunks(t)) for s, t in
        ((s, s.themes()) for s in fs.graph.shelves(facet="foods"))]
rows = sorted([(s, t, cs) for s, t, cs in rows if t], key=lambda x: len(x[2]), reverse=True)
overview = pd.DataFrame([{"shelf_id": s.shelf_id, "label": s.label,
                          "themed_chunks": len(cs), "themes": len(t),
                          "subtree_chunks": s.chunk_count} for s, t, cs in rows])
display(overview.head(15))

SHELF_ID = None  # set to a shelf_id from the table above (e.g. a tight dairy / cereal-grain domain)
eligible = [x for x in rows if len(x[1]) >= 3] or rows   # need ≥3 themes for a meaningful partition
chosen = next((x for x in rows if x[0].shelf_id == SHELF_ID), eligible[0]) if SHELF_ID else eligible[0]
shelf, themes = chosen[0], chosen[1]
print(f"\nchosen: '{shelf.label}'  ({len(chosen[2])} themed chunks, {len(themes)} Layer B themes)")

,shelf_id,label,themed_chunks,themes,subtree_chunks
0,foodon:FOODON:03316531,fruit produce,630,18,633
1,foodon:FOODON:03315150,mammalian milk product,619,20,649
2,foodon:FOODON:00003816,vegetable,538,18,542
3,foodon:FOODON:03306429,meat (raw),472,16,478
4,foodon:FOODON:00001257,milk or milk based food product,347,13,1076
5,foodon:FOODON:00001185,rice food product,256,14,257
6,foodon:FOODON:03301367,nuts (mixed),246,13,250
7,foodon:FOODON:03310646,legume,243,13,546
8,foodon:FOODON:03000287,cheese,235,14,239
9,foodon:FOODON:00001917,grain based bakery food product,219,13,226



chosen: 'fruit produce'  (630 themed chunks, 18 Layer B themes)


In [4]:
# Build the Layer B partition over the shelf's chunks: chunk_id -> theme (first membership; -1 = unthemed).
shelf_chunks = shelf.chunks()
chunk_text = {c.chunk_id: c.text for c in shelf_chunks}
lb_theme = {}
theme_label = {}
for ti, th in enumerate(themes):
    theme_label[ti] = th.label
    for cid in fs.graph_store.get_chunks_for_theme(th.theme_id):
        lb_theme.setdefault(cid, ti)   # first theme wins if multi-membership

# Common chunk set: on the shelf, has text, has a cached vector.
kept = [c.chunk_id for c in shelf_chunks if c.chunk_id in id2vec and c.chunk_id in chunk_text]
docs = [chunk_text[i] for i in kept]
X = np.asarray([id2vec[i] for i in kept])
lb_labels = np.asarray([lb_theme.get(i, -1) for i in kept])
K_lb = len(themes)
print(f"shelf chunks usable: {len(kept)} | Layer B themed: {np.mean(lb_labels != -1):.1%} "
      f"| themes (K): {K_lb}")

shelf chunks usable: 633 | Layer B themed: 99.5% | themes (K): 18


## 4 · Cluster the same chunks with BERTopic

If a provider key is in the env (same one §2 uses), BERTopic also names its topics with an LLM via a
`representation_model` (BERTopic's "6B. LLM & Generative AI") — so **both sides are labeled the same way**.
This is cheap here (~K calls, one shelf). Without a key it uses c-TF-IDF keywords. Labels don't affect
the clustering or any agreement number — they only make §5/§6 readable.

In [5]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.dimensionality import BaseDimensionalityReduction

CLUSTERER = "kmeans_matched"   # "kmeans_matched" | "agglomerative_matched" | "hdbscan"

K_eff = max(2, min(K_lb, len(docs) - 1))   # never ask for more clusters than points
if CLUSTERER == "kmeans_matched":
    reducer = BaseDimensionalityReduction()                 # cluster the raw BGE vectors, matched K
    cluster_model = KMeans(n_clusters=K_eff, random_state=RANDOM_STATE, n_init=10)
elif CLUSTERER == "agglomerative_matched":
    reducer = BaseDimensionalityReduction()
    cluster_model = AgglomerativeClustering(n_clusters=K_eff)
else:  # "hdbscan" — density clustering DEGENERATES on raw 768-d vectors, so it keeps BERTopic's UMAP step
    from umap import UMAP
    from hdbscan import HDBSCAN
    reducer = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE)
    cluster_model = HDBSCAN(min_cluster_size=15, metric="euclidean",
                            cluster_selection_method="eom", prediction_data=True)

# Optional LLM topic labels — uses the SAME provider/key/model resolved in §1, so both sides are
# named the same way. Only ~K calls here (one shelf) vs Layer B's ~600.
# OpenRouter / Groq / OpenAI are OpenAI-compatible; Anthropic on this side needs bertopic's LiteLLM.
representation_model = None
_prompt = ("I have a topic described by these documents:\n[DOCUMENTS]\n"
           "and these keywords: [KEYWORDS]\nGive a concise (<=6 words) label.\n"
           "Based on the information above, extract a short topic label in the following format:\n"
           "topic: <label>")
_base_url = {"openrouter": "https://openrouter.ai/api/v1",
             "groq": "https://api.groq.com/openai/v1", "openai": None}
if LLM_ENABLED and LLM_PROVIDER in _base_url:
    try:
        import openai
        from bertopic.representation import OpenAI as BTOpenAI
        _kw = {"api_key": os.environ[_ENV_VAR]}
        if _base_url[LLM_PROVIDER]:
            _kw["base_url"] = _base_url[LLM_PROVIDER]
        # doc_length needs a tokenizer ("char"/"whitespace"/"vectorizer") to count against.
        representation_model = BTOpenAI(openai.OpenAI(**_kw), model=LLM_MODEL, chat=True,
                                        nr_docs=4, doc_length=400, tokenizer="char",
                                        delay_in_seconds=1, prompt=_prompt)
        print(f"BERTopic LLM labels via {LLM_PROVIDER}/{LLM_MODEL}")
    except Exception as e:
        print("BERTopic LLM labels unavailable, using c-TF-IDF:", e)
elif LLM_ENABLED and LLM_PROVIDER == "anthropic":
    print("BERTopic: Anthropic needs LiteLLM on this side — using c-TF-IDF labels (Layer B still LLM)")
else:
    print("BERTopic: no LLM key → c-TF-IDF keyword labels (set LLM_API_KEY in §1)")

bt = BERTopic(
    umap_model=reducer,                            # passthrough for matched clusterers; UMAP for HDBSCAN
    hdbscan_model=cluster_model,                   # the swappable clusterer (per BERTopic docs)
    # min_df=1: c-TF-IDF runs over the K per-cluster documents, so min_df must be ≤ K (small per shelf)
    vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1),
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    representation_model=representation_model,      # None → c-TF-IDF; else LLM-written labels
    calculate_probabilities=False, verbose=False,
)
try:
    bt_topics = np.asarray(bt.fit_transform(docs, embeddings=X)[0])
except Exception as e:
    if representation_model is None:
        raise
    # LLM labeling failed during fit (bad model id, API/ratelimit) — refit with c-TF-IDF labels.
    print("⚠ LLM labeling failed during fit — refitting with c-TF-IDF labels.\n  ", repr(e)[:300])
    bt = BERTopic(
        umap_model=reducer, hdbscan_model=cluster_model,
        vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        calculate_probabilities=False, verbose=False,
    )
    bt_topics = np.asarray(bt.fit_transform(docs, embeddings=X)[0])
print(f"BERTopic ({CLUSTERER}): {len(set(bt_topics) - {-1})} topics | "
      f"coverage {np.mean(bt_topics != -1):.1%}")

BERTopic LLM labels via groq/llama-3.1-8b-instant


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completion

BERTopic (kmeans_matched): 18 topics | coverage 100.0%


## 5 · Compare the two partitions

In [6]:
from sklearn.metrics import (normalized_mutual_info_score as NMI,
                             adjusted_rand_score as ARI, v_measure_score)

def top_words_per_cluster(docs, labels, n=10):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
    M = vec.fit_transform(docs); terms = np.array(vec.get_feature_names_out())
    out = {}
    for c in sorted(set(labels) - {-1}):
        rows = np.where(labels == c)[0]
        if len(rows) == 0: continue
        mean = np.asarray(M[rows].mean(axis=0)).ravel()
        out[c] = list(terms[mean.argsort()[::-1][:n]])
    return out

def npmi(tw, docs):
    from sklearn.feature_extraction.text import CountVectorizer as CV
    vocab = sorted({w for ws in tw.values() for w in ws})
    if not vocab: return float("nan")
    Xb = CV(vocabulary=vocab, binary=True, ngram_range=(1, 2)).fit_transform(docs).tocsc()
    D = Xb.shape[0]; col = {w: j for j, w in enumerate(vocab)}
    dfc = np.asarray(Xb.sum(axis=0)).ravel()
    def _n(a, b):
        da, db = dfc[col[a]], dfc[col[b]]
        if da == 0 or db == 0: return 0.0
        co = Xb[:, col[a]].multiply(Xb[:, col[b]]).nnz
        if co == 0: return -1.0
        pa, pb, pab = da/D, db/D, co/D
        return math.log(pab/(pa*pb)) / (-math.log(pab))
    vals = [np.mean([_n(a, b) for a, b in combinations(ws, 2)]) for ws in tw.values() if len(ws) > 1]
    return float(np.nanmean(vals)) if vals else float("nan")

# core mask = chunks assigned by BOTH methods (drop either-side -1) for a clean agreement read
core = (lb_labels != -1) & (bt_topics != -1)
lb_tw = top_words_per_cluster(docs, lb_labels)
bt_tw = top_words_per_cluster(docs, bt_topics)

def agree(fn):  # guard: agreement is undefined if either side collapsed to <2 clusters on the overlap
    ok = core.sum() >= 2 and len(set(lb_labels[core])) >= 2 and len(set(bt_topics[core])) >= 2
    return f"{fn(lb_labels[core], bt_topics[core]):.3f}" if ok else "n/a (degenerate)"

summary = pd.DataFrame([
    ("usable chunks",               len(kept)),
    ("Layer B themes / coverage",  f"{K_lb} / {np.mean(lb_labels != -1):.1%}"),
    ("BERTopic clusters / coverage", f"{len(set(bt_topics) - {-1})} / {np.mean(bt_topics != -1):.1%}"),
    ("agreement ARI (core)",       agree(ARI)),
    ("agreement NMI (core)",       agree(NMI)),
    ("agreement V-measure (core)", agree(v_measure_score)),
    ("NPMI coherence — Layer B",   f"{npmi(lb_tw, docs):.3f}"),
    ("NPMI coherence — BERTopic",  f"{npmi(bt_tw, docs):.3f}"),
], columns=["metric", "value"])
summary

,metric,value
0,usable chunks,633
1,Layer B themes / coverage,18 / 99.5%
2,BERTopic clusters / coverage,18 / 100.0%
3,agreement ARI (core),0.299
4,agreement NMI (core),0.500
5,agreement V-measure (core),0.500
6,NPMI coherence — Layer B,0.091
7,NPMI coherence — BERTopic,0.155


In [7]:
# Best-match table: each Layer B theme ↔ the BERTopic cluster it overlaps most (chunk Jaccard).
def members(labels, c): return set(np.where(labels == c)[0])
rows = []
for ti in sorted(set(lb_labels) - {-1}):
    A = members(lb_labels, ti)
    best, bestj = None, 0.0
    for cj in sorted(set(bt_topics) - {-1}):
        B = members(bt_topics, cj)
        j = len(A & B) / len(A | B) if (A | B) else 0.0
        if j > bestj: best, bestj = cj, j
    rows.append({
        "layerB_theme": theme_label.get(ti, ti),
        "size": len(A),
        "jaccard": round(bestj, 2),
        "bertopic_words": ", ".join((bt_tw.get(best) or [])[:6]) if best is not None else "—",
    })
pd.DataFrame(rows).sort_values("size", ascending=False)

,layerB_theme,size,jaccard,bertopic_words
0,preparing and storing food safely,79,0.36,"food, irradiation, wash, foods, produce, h18567"
1,carbohydrate metabolism and sugars,75,0.40,"carbohydrates, sugars, carbohydrate, sugar, ad..."
2,reducing cancer risk with diet,62,0.41,"fiber, cancer, risk, disease, heart, health"
3,nutrition for growing children,60,0.60,"children, child, milk, school, foods, offer"
4,daily fruit and vegetable servings,59,0.21,"servings, servings day, day, cup, rice, pasta"
5,eating for a sustainable future,54,0.40,"environmental, food, sustainable, impact, cons..."
6,healthy snack options,52,0.17,"servings, servings day, day, cup, rice, pasta"
7,plant based dietary patterns,36,0.35,"vegetarian, vegetarians, diet, foods, teen, ea..."
8,eating nutrient-dense foods for weight,34,0.62,"fat, weight, calories, foods, choose, healthy"
9,nutrition during pregnancy for women,29,0.48,"pregnancy, women, pregnant, extra, pregnant wo..."


## 6 · BERTopic topic tree for this shelf

The hierarchical view from the official BERTopic notebook (`get_topic_tree` + `visualize_hierarchy`).
BERTopic agglomerates its topics' c-TF-IDF profiles into a tree — a **within-shelf hierarchy** that
Layer B doesn't produce (its themes are flat per shelf). Hover the merge nodes in the plot to see the
combined representation at each level.

In [ ]:
OUT = ROOT / "notebooks" / "out"; OUT.mkdir(parents=True, exist_ok=True)
hier = bt.hierarchical_topics(docs)
print(bt.get_topic_tree(hier))
fig = bt.visualize_hierarchy(hierarchical_topics=hier)
fig.write_html(str(OUT / f"headtohead_{shelf.shelf_id.replace(':', '_').replace('/', '_')}_hierarchy.html"))
fig

## 7 · FoodScholar shelf + theme tree (Layer A + Layer B)

The native FoodScholar view, exactly as `graph_build.ipynb` renders it: the full **Layer A shelf
hierarchy** for the facet, with each shelf's **Layer B themes** grouped by discovery origin
(Merged / Similarity / Relatedness). Click a shelf on the left to see its themes on the right. This is
the *shelves-and-topics* tree — contrast its food-axis backbone against BERTopic's concept-axis tree
in §6.

In [ ]:
from IPython.display import IFrame
VIZ = ROOT / "data" / "viz"; VIZ.mkdir(parents=True, exist_ok=True)
out = fs.viz.layer_a_tree("foods").render("tree", output=VIZ / "layer_a_tree_foods.html")
print("wrote", out)
# IFrame src is relative to this notebook (in notebooks/), hence the ../.
IFrame(src="../data/viz/layer_a_tree_foods.html", width="100%", height=720)

## 8 · Per-shelf chunk explorer — Layer B themes ↔ BERTopic topics

A self-contained HTML for the **selected shelf**: left = Layer B themes, right = BERTopic's **flat
(non-hierarchical) topics from §4** — each row expands to the **chunks inside it**. Layer B chunks
carry a **direct / indirect** badge: *direct* = text-similarity core (≈ Pass 1 — cosine to the theme
centroid ≥ the Pass-1 edge threshold), *indirect* = held by entity relatedness (≈ Pass 2 — in the
theme but not text-coherent). Uses native `<details>` (no JS) + the WiseFood palette. Saved to `data/viz/`.

In [ ]:
import html as _h
from IPython.display import IFrame

PASS1_THR = fs.config.layer_b.similarity.edge_threshold   # 0.55 — the Pass-1 cosine floor
meta = {c.chunk_id: (c.text, c.source_doc_id) for c in shelf_chunks}

def _cos(a, b):
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0

def esc(s): return _h.escape("" if s is None else str(s))
def snip(t, n=220):
    t = " ".join((t or "").split())
    return esc(t[:n]) + ("…" if len(t) > n else "")

# Layer B: each theme → member chunks tagged by WHICH PASS linked them:
#   similarity theme → all direct (text core); relatedness theme → all indirect (entity link);
#   merged theme → per-chunk (direct iff it has a text neighbour ≥ pass-1 threshold).
lb_cards = []
for th in themes:
    cids = list(fs.graph_store.get_chunks_for_theme(th.theme_id))
    dp = th.model.discovery_pass
    vmap = {c: id2vec[c] for c in cids if c in id2vec}

    def _badge(c, dp=dp, vmap=vmap):
        if dp == "global_similarity":
            return "direct"
        if dp == "relatedness":
            return "indirect"
        v = vmap.get(c)
        if v is None:
            return "indirect"
        best = max((_cos(v, w) for cc, w in vmap.items() if cc != c), default=0.0)
        return "direct" if best >= PASS1_THR else "indirect"

    rows = []
    for c in cids:
        txt, doc = meta.get(c, (chunk_text.get(c, ""), "?"))
        rows.append((_badge(c), txt, doc))
    rows.sort(key=lambda r: r[0] != "direct")   # direct first
    lb_cards.append((th.label, dp, rows))

# BERTopic: each flat topic → its member chunks.
bt_names = bt.get_topic_info().set_index("Topic")["Name"].to_dict()
bt_cards = []
for t in sorted(set(bt_topics) - {-1}):
    idxs = [i for i in range(len(kept)) if bt_topics[i] == t]
    words = ", ".join(w for w, _ in (bt.get_topic(t) or [])[:6])
    rows = [(chunk_text.get(kept[i], ""), meta.get(kept[i], ("", "?"))[1]) for i in idxs]
    bt_cards.append((bt_names.get(t, str(t)), words, rows))

CSS = '''<style>
:root{--brand:#d53355;--brandg:#646d1a;--terra:#b5663f;--header:#173f35;
--panel:#f6efe6;--line:#e8e1d4;--muted:#7a6657;--fg:#1f1a15;}
*{box-sizing:border-box}body{margin:0;font:14px/1.45 "Quicksand",system-ui,sans-serif;color:var(--fg)}
header{background:var(--header);color:#f6efe6;padding:12px 18px;font-size:15px}
.cols{display:flex;gap:0}.col{flex:1;padding:14px 16px;overflow:auto;height:calc(100vh - 90px)}
.col:first-child{border-right:1px solid var(--line)}
h3{margin:0 0 10px;font-size:14px;color:var(--muted);text-transform:uppercase;letter-spacing:.4px}
details{background:var(--panel);border-radius:10px;margin:6px 0;padding:6px 10px;border-left:3px solid var(--brandg)}
.col:last-child details{border-left-color:var(--brand)}
summary{cursor:pointer;font-weight:600}summary .meta{color:var(--muted);font-weight:400;font-size:12px;margin-left:6px}
.kw{color:var(--muted);font-size:12px;margin:4px 0 6px}
.chunk{border-top:1px solid var(--line);padding:6px 2px}
.src{color:var(--muted);font-size:11px}.tx{font-size:12.5px;margin-top:2px}
.b{font-size:10.5px;font-weight:700;border-radius:999px;padding:1px 7px;margin-right:6px}
.b.direct{background:#edf0d5;color:#424811}.b.indirect{background:#f3e0d6;color:#7a3f24}
.b.\\2014{background:#eee;color:#999}
</style>'''

P = ["<!DOCTYPE html><html><head><meta charset='utf-8'>", CSS, "</head><body>"]
P.append(f"<header><b>{esc(shelf.label)}</b> — Layer B themes (dual-pass) vs BERTopic flat topics ({CLUSTERER})</header>")
P.append("<div class='cols'><div class='col'><h3>Layer B themes — Leiden dual-pass</h3>")
for label, dpass, rows in lb_cards:
    nd = sum(1 for r in rows if r[0] == "direct"); ni = sum(1 for r in rows if r[0] == "indirect")
    P.append(f"<details><summary>{esc(label)}<span class='meta'>{len(rows)} chunks · {nd} direct / {ni} indirect · {esc(dpass)}</span></summary>")
    for badge, txt, doc in rows:
        P.append(f"<div class='chunk'><span class='b {badge}'>{badge}</span>"
                 f"<span class='src'>{esc(doc)}</span><div class='tx'>{snip(txt)}</div></div>")
    P.append("</details>")
P.append("</div><div class='col'><h3>BERTopic topics — flat / non-hierarchical</h3>")
for name, words, rows in bt_cards:
    P.append(f"<details><summary>{esc(name)}<span class='meta'>{len(rows)} chunks</span></summary>"
             f"<div class='kw'>{esc(words)}</div>")
    for txt, doc in rows:
        P.append(f"<div class='chunk'><span class='src'>{esc(doc)}</span><div class='tx'>{snip(txt)}</div></div>")
    P.append("</details>")
P.append("</div></div></body></html>")

VIZ = ROOT / "data" / "viz"; VIZ.mkdir(parents=True, exist_ok=True)
fn = VIZ / f"headtohead_chunks_{shelf.shelf_id.replace(':', '_').replace('/', '_')}.html"
fn.write_text("".join(P), encoding="utf-8")
print("wrote", fn)
IFrame(src=f"../data/viz/{fn.name}", width="100%", height=720)

## 9 · Theme ↔ chunk ↔ topic linkage

Tri-pane: **Layer B themes** (left) · **shared chunks** (middle) · **BERTopic topics** (right). Each
chunk belongs to one theme *and* one topic, so the chunk is the link. Click a theme → its chunks fill
the middle and each topic shows how many it shares (`·N∩`, highlighted); click a topic too → the middle
narrows to the **common chunks** of that theme×topic. Chunk badges mark direct (Pass-1) / indirect
(Pass-2). Click a theme/topic name inside a chunk to jump to it. Saved to `data/viz/`.

In [ ]:
import json
import html as _h2
from IPython.display import IFrame

PASS1_THR = fs.config.layer_b.similarity.edge_threshold
def _cos3(a, b):
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0
def _esc3(s): return _h2.escape("" if s is None else str(s))
def _snip3(t, n=360):
    t = " ".join((t or "").split())
    return t[:n] + ("…" if len(t) > n else "")

# Per-chunk direct/indirect (pass-aware, same rule as §8 / the library tree).
chunk_badge = {}
for ti, th in enumerate(themes):
    cids = list(fs.graph_store.get_chunks_for_theme(th.theme_id))
    dp = th.model.discovery_pass
    vmap = {c: id2vec[c] for c in cids if c in id2vec}
    for c in cids:
        if dp == "global_similarity":
            chunk_badge[c] = "direct"
        elif dp == "relatedness":
            chunk_badge[c] = "indirect"
        else:
            v = vmap.get(c)
            chunk_badge[c] = ("indirect" if v is None else
                ("direct" if max((_cos3(v, w) for cc, w in vmap.items() if cc != c), default=0.0) >= PASS1_THR
                 else "indirect"))

meta3 = {c.chunk_id: (c.text, c.source_doc_id) for c in shelf_chunks}
CH = []
for i, cid in enumerate(kept):
    txt, doc = meta3.get(cid, (chunk_text.get(cid, ""), "?"))
    CH.append({"id": cid, "t": int(lb_labels[i]), "k": int(bt_topics[i]),
               "b": chunk_badge.get(cid, "—"), "s": _snip3(txt), "src": doc})
bt_names = bt.get_topic_info().set_index("Topic")["Name"].to_dict()
THEMES_J = [{"i": ti, "label": theme_label.get(ti, str(ti))}
            for ti in sorted({int(x) for x in lb_labels} - {-1})]
TOPICS_J = [{"i": int(t), "label": bt_names.get(t, str(t))}
            for t in sorted({int(x) for x in bt_topics} - {-1})]

CSS = '''<style>
:root{--brand:#d53355;--brandg:#646d1a;--header:#173f35;--panel:#f6efe6;--line:#e8e1d4;
--muted:#7a6657;--fg:#1f1a15;--sel:#f7d6dd;--hl:#edf0d5;}
*{box-sizing:border-box}body{margin:0;font:14px/1.45 "Quicksand",system-ui,sans-serif;color:var(--fg)}
header{background:var(--header);color:#f6efe6;padding:10px 16px;display:flex;gap:12px;align-items:center}
header button{margin-left:auto;background:#f6efe6;border:0;border-radius:8px;padding:4px 12px;cursor:pointer}
.cols{display:flex;height:calc(100vh - 80px)}
.col{flex:1;overflow:auto;padding:12px 14px;border-right:1px solid var(--line)}
.col.mid{flex:1.4;background:var(--panel)}.col:last-child{border-right:0}
h3{margin:0 0 10px;font-size:13px;color:var(--muted);text-transform:uppercase;letter-spacing:.4px}
.item{padding:6px 9px;border-radius:8px;cursor:pointer;display:flex;gap:8px;align-items:baseline}
.item:hover{background:var(--hl)}.item.on{background:var(--sel)}.item.hl{outline:1px solid var(--brandg)}
.item .n{margin-left:auto;color:var(--brandg);font-variant-numeric:tabular-nums;font-size:12px}
.chunk{background:#fff;border:1px solid var(--line);border-radius:8px;padding:7px 9px;margin:6px 0}
.b{font-size:10px;font-weight:700;border-radius:999px;padding:1px 7px;margin-right:6px}
.b.direct{background:#edf0d5;color:#424811}.b.indirect{background:#f3e0d6;color:#7a3f24}
.lnk{cursor:pointer;font-size:12px;font-weight:600;color:var(--brand)}.lnk:hover{text-decoration:underline}
.src{color:var(--muted);font-size:11px;margin-left:6px}.tx{font-size:12.5px;margin-top:3px}
.empty{color:var(--muted);font-style:italic}
</style>'''

JS_DATA = ("<script>\nconst CH=" + json.dumps(CH) + ";\nconst THEMES=" + json.dumps(THEMES_J) +
           ";\nconst TOPICS=" + json.dumps(TOPICS_J) + ";\n")
JS_BODY = r'''
let selT=null, selK=null;
const TH=document.getElementById("themes"),TP=document.getElementById("topics"),
      MID=document.getElementById("chunks"),MH=document.getElementById("midhead");
function esc(s){return String(s==null?"":s).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");}
function tl(i){const t=THEMES.find(x=>x.i===i);return t?t.label:"—";}
function kl(i){const t=TOPICS.find(x=>x.i===i);return t?t.label:"—";}
function render(){
  TH.innerHTML=THEMES.map(t=>{const n=CH.filter(c=>c.t===t.i).length;
    const ov=(selK!==null)?CH.filter(c=>c.t===t.i&&c.k===selK).length:0;
    const cls=(selT===t.i?"on":"")+((selK!==null&&ov)?" hl":"");
    return `<div class="item ${cls}" onclick="pickT(${t.i})">${esc(t.label)}<span class="n">${n}${ov?` · ${ov}∩`:""}</span></div>`;}).join("");
  TP.innerHTML=TOPICS.map(t=>{const n=CH.filter(c=>c.k===t.i).length;
    const ov=(selT!==null)?CH.filter(c=>c.k===t.i&&c.t===selT).length:0;
    const cls=(selK===t.i?"on":"")+((selT!==null&&ov)?" hl":"");
    return `<div class="item ${cls}" onclick="pickK(${t.i})">${esc(t.label)}<span class="n">${n}${ov?` · ${ov}∩`:""}</span></div>`;}).join("");
  let cs=CH;
  if(selT!==null&&selK!==null) cs=CH.filter(c=>c.t===selT&&c.k===selK);
  else if(selT!==null) cs=CH.filter(c=>c.t===selT);
  else if(selK!==null) cs=CH.filter(c=>c.k===selK);
  MH.textContent=`${cs.length} chunk(s)`+(selT!==null?` · theme: ${tl(selT)}`:"")+(selK!==null?` · topic: ${kl(selK)}`:"");
  MID.innerHTML=cs.map(c=>`<div class="chunk"><span class="b ${c.b}">${c.b}</span>`+
    `<span class="lnk" onclick="pickT(${c.t})">▸ ${esc(tl(c.t))}</span> `+
    `<span class="lnk" onclick="pickK(${c.k})">${esc(kl(c.k))} ◂</span>`+
    `<span class="src">${esc(c.src)}</span><div class="tx">${esc(c.s)}</div></div>`).join("")||`<p class="empty">No chunks for this selection.</p>`;
}
function pickT(i){selT=(selT===i?null:i);render();}
function pickK(i){selK=(selK===i?null:i);render();}
function clearSel(){selT=null;selK=null;render();}
render();
</script>'''

BODY = (f"<header><b>{_esc3(shelf.label)}</b> — themes ↔ chunks ↔ topics"
        "<button onclick=\"clearSel()\">clear</button></header>"
        "<div class='cols'>"
        "<div class='col'><h3>Layer B themes</h3><div id='themes'></div></div>"
        "<div class='col mid'><h3 id='midhead'>common chunks</h3><div id='chunks'></div></div>"
        "<div class='col'><h3>BERTopic topics</h3><div id='topics'></div></div></div>")
page = ("<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'><title>" +
        _esc3(shelf.label) + "</title>" + CSS + "</head><body>" + BODY + JS_DATA + JS_BODY +
        "</body></html>")

VIZ = ROOT / "data" / "viz"; VIZ.mkdir(parents=True, exist_ok=True)
fn = VIZ / f"linkage_{shelf.shelf_id.replace(':', '_').replace('/', '_')}.html"
fn.write_text(page, encoding="utf-8")
print("wrote", fn)
IFrame(src=f"../data/viz/{fn.name}", width="100%", height=720)

## 10 · Secondary — native HDBSCAN granularity & outliers

In [ ]:
# BERTopic the way it ships (UMAP + HDBSCAN): shows natural cluster count and the outlier bucket
# that Layer B avoids by construction.
from umap import UMAP
from hdbscan import HDBSCAN
native = BERTopic(
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE),
    hdbscan_model=HDBSCAN(min_cluster_size=15, metric="euclidean", cluster_selection_method="eom"),
    vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1),
    calculate_probabilities=False, verbose=False,
)
nt = np.asarray(native.fit_transform(docs, embeddings=X)[0])
pd.DataFrame([
    ("Layer B (Leiden, dual-pass)", K_lb, f"{np.mean(lb_labels != -1):.1%}"),
    ("BERTopic native (UMAP+HDBSCAN)", len(set(nt) - {-1}), f"{np.mean(nt != -1):.1%}"),
], columns=["method", "clusters", "coverage"])

## 11 · Reading the head-to-head

- **Agreement (ARI / NMI / V on the core)** — with matched K and the same vectors, this is *how
  similarly* Leiden and k-Means carve the shelf. High ⇒ on a single food domain the entity signal
  barely changes the cut (text already separates the sub-topics); moderate/low ⇒ **Pass 2's FoodOn
  co-occurrence pulls the boundaries somewhere text alone wouldn't** — look at the best-match table
  for the themes with low Jaccard to see which ones.
- **Coherence (NPMI)** — same method applied to both partitions, so it's a fair quality read. Similar
  scores mean BERTopic could stand in for Pass-1 clustering on coherence grounds.
- **Coverage & granularity (§6)** — native BERTopic picks its own K and drops outliers; Layer B hits
  a per-shelf target with full coverage. This is the contract difference, not a quality one — and
  matched k-Means (§4) shows BERTopic *can* honor the same contract when asked.

**So, for "make Layer B better":** if agreement is high and coherence comparable, BERTopic's
clusterer is a viable **opt-in Pass-1 backend** (you'd keep Pass 2 + merge, and use `reduce_outliers`
or k-Means for coverage). If agreement is low, the divergence is exactly the entity-driven value Pass 2
adds — and the case for *keeping* the dual pass rather than replacing it.

## 12 · BERTopic-native theme rollup — leaf themes → a hierarchy *across* children

The question: can **BERTopic alone** take the per-shelf leaf themes and build a **theme hierarchy across
sibling child shelves** — i.e. discover parent-level themes that group related child themes? BERTopic
does *both* levels here; no sklearn agglomeration, no Leiden.

1. **Leaf level** — each themed child shelf gets its **own BERTopic** fit (matched-K KMeans on the
   cached BGE vectors, same recipe as §4) → its leaf themes, each with BERTopic's representative docs.
2. **Parent level** — pool every leaf theme into a single "theme document" (its representative docs,
   tagged with the child shelf it came from) and fit **one BERTopic over those theme-documents**. Then
   call **`hierarchical_topics()` + `get_topic_tree()`** on it — BERTopic agglomerates the leaf themes
   into a dendrogram. **That tree is the rollup, produced natively by BERTopic.**

We read off the cross-child structure: which leaf themes BERTopic merges, and whether a merged
parent-topic **spans multiple child shelves** (the cross-shelf grouping per-shelf Layer B never makes).
§13 renders this tree side by side with the child shelves that fed it.

> This is the corrected version of the earlier draft: the rollup is done by **BERTopic's own
> `hierarchical_topics`**, not by hand-rolled clustering, and it is framed as *producing a theme
> hierarchy across children* — not as a speed optimization (chunks attach to their nearest ancestor
> only, so there is no big-ancestor re-cluster to avoid; that idea was a dead end and is dropped).


In [ ]:
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

# ── pick an inner parent shelf with ≥2 themed leaf children (skip the synthetic facet root) ────────
def _theme_count(sh):
    return len(sh.themes())

SYNTH_ROOT = "facet:foods"
foods = fs.graph.shelves(facet="foods")

def _own_chunk_ids(sh):
    own = {c.chunk_id for c in sh.chunks()}
    for gk in sh.children():
        own -= {c.chunk_id for c in gk.chunks()}
    return own

inner = []
for s in foods:
    if s.shelf_id == SYNTH_ROOT:
        continue
    leaf_themed = [k for k in s.children() if _theme_count(k) > 0 and not k.children()]
    if len(leaf_themed) >= 2:
        inner.append((s, leaf_themed, sum(len(_own_chunk_ids(k)) for k in leaf_themed)))
inner.sort(key=lambda x: x[2], reverse=True)
assert inner, "no eligible inner shelf (need ≥2 themed leaf children)"

PARENT_ID = None
P, kids_themed, reusable = (next((x for x in inner if x[0].shelf_id == PARENT_ID), inner[0])
                            if PARENT_ID else inner[0])
print(f"parent shelf: '{P.label}'  ({P.shelf_id})  |  {len(kids_themed)} themed leaf children")
print("  top eligible: " + ", ".join(f"{x[0].label}({len(x[1])})" for x in inner[:5]))

def _vecs(cids):
    cc = [c for c in cids if c in id2vec]
    return cc, (np.asarray([id2vec[c] for c in cc]) if cc else np.empty((0, emb.shape[1])))

def _text_for(cids):
    return {c.chunk_id: c.text for c in fs.chunk_store.get_many(list(cids))}

_REDUCER = BaseDimensionalityReduction()   # passthrough — cluster the raw BGE vectors (same as §4)
_VEC = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
_CTFIDF = ClassTfidfTransformer(reduce_frequent_words=True)

# ── (1) LEAF LEVEL — one BERTopic per themed child shelf → leaf themes + representative docs ───────
leaf_docs = []        # one "theme document" per leaf theme (its representative docs joined)
leaf_shelf = []       # child shelf label for each leaf theme
leaf_label = []       # BERTopic's name for each leaf theme
leaf_members = []     # chunk-id set per leaf theme (for §13 / counts)
for k in kids_themed:
    cc, M = _vecs(_own_chunk_ids(k))
    if len(cc) < 3:
        continue
    txt = _text_for(cc)
    docs_k = [txt[c] for c in cc]
    Kc = max(2, min(_theme_count(k), len(cc) - 1))
    bt_k = BERTopic(umap_model=_REDUCER, hdbscan_model=KMeans(n_clusters=Kc, random_state=RANDOM_STATE, n_init=10),
                    vectorizer_model=_VEC, ctfidf_model=_CTFIDF,
                    calculate_probabilities=False, verbose=False)
    tk = np.asarray(bt_k.fit_transform(docs_k, embeddings=M)[0])
    info = bt_k.get_topic_info().set_index("Topic")
    rep = info["Representative_Docs"].to_dict() if "Representative_Docs" in info else {}
    names = info["Name"].to_dict()
    for t in sorted(set(tk) - {-1}):
        idx = [i for i in range(len(cc)) if tk[i] == t]
        rep_docs = rep.get(t) or [docs_k[i] for i in idx[:4]]
        leaf_docs.append(" \n".join(str(d) for d in rep_docs))
        leaf_shelf.append(k.label)
        leaf_label.append(f"[{k.label}] {names.get(t, t)}")
        leaf_members.append({cc[i] for i in idx})
n_leaf = len(leaf_docs)
print(f"\nleaf level: {n_leaf} leaf themes across {len({s for s in leaf_shelf})} child shelves")

# ── (2) PARENT LEVEL — ONE BERTopic over the leaf-theme documents, then hierarchical_topics() ──────
# Each leaf theme is ONE document here. BERTopic clusters them (matched-K) and then agglomerates the
# flat parent-topics into a dendrogram ACROSS children — the rollup, produced by BERTopic itself.
leaf_emb = np.stack([  # embed each theme-doc as the unit-mean of its member chunk vectors
    (lambda v: v / (np.linalg.norm(v) or 1.0))(np.mean([id2vec[c] for c in mem if c in id2vec] or
                                                        [np.zeros(emb.shape[1])], axis=0))
    for mem in leaf_members
])
K_parent = max(2, min(_theme_count(P) or len(kids_themed), n_leaf - 1))
bt_parent = BERTopic(umap_model=_REDUCER, hdbscan_model=KMeans(n_clusters=K_parent, random_state=RANDOM_STATE, n_init=10),
                     vectorizer_model=_VEC, ctfidf_model=_CTFIDF,
                     calculate_probabilities=False, verbose=False)
parent_topics = np.asarray(bt_parent.fit_transform(leaf_docs, embeddings=leaf_emb)[0])

# BERTopic's native hierarchy over the leaf themes:
hier = bt_parent.hierarchical_topics(leaf_docs)
print("\nBERTopic theme tree (across children):")
print(bt_parent.get_topic_tree(hier))

# flat parent-topic → which leaf themes (and which child shelves) it grouped
ptopic_leaves = {}
for li, pt in enumerate(parent_topics):
    ptopic_leaves.setdefault(int(pt), []).append(li)
summary = []
for pt, lis in sorted(ptopic_leaves.items()):
    shelves_in = sorted({leaf_shelf[li] for li in lis})
    summary.append({
        "parent_topic": bt_parent.get_topic_info().set_index("Topic")["Name"].get(pt, pt),
        "leaf_themes": len(lis),
        "child_shelves_spanned": len(shelves_in),
        "shelves": ", ".join(s[:18] for s in shelves_in),
    })
display(pd.DataFrame(summary).sort_values("leaf_themes", ascending=False).reset_index(drop=True))
n_cross = sum(1 for r in summary if r["child_shelves_spanned"] > 1)
print(f"\n{n_cross}/{len(summary)} parent topics span >1 child shelf "
      f"(cross-shelf theme groupings BERTopic found that per-shelf Layer B cannot).")


## 13 · Side-by-side — BERTopic's theme hierarchy ↔ the child shelves it grouped

Cell 9 stayed inside one shelf. This spans the **whole parent subtree** and renders **BERTopic's own
rollup** from §12. Left = the **parent topics BERTopic formed over the leaf themes** (each expands to
the leaf themes it grouped, tagged by source child shelf); right = the **child shelves** and their leaf
themes. The link is BERTopic's grouping: **clicking a parent topic highlights the contributing child
shelves/themes** on the right, and flags topics that **span >1 child shelf** — the cross-shelf theme
groupings BERTopic discovers that per-shelf Layer B cannot. The native `get_topic_tree` text dendrogram
from §12 is the same structure in ASCII; this is the interactive, shelf-aware view. Saved to `data/viz/`.


In [ ]:
import json as _json
import html as _hh
from IPython.display import IFrame

# child shelf order + index (right pane)
child_shelf_order = sorted({s for s in leaf_shelf})
shelf_idx = {lab: si for si, lab in enumerate(child_shelf_order)}

# Left pane: BERTopic parent topics, each with the leaf themes (and child shelves) it grouped.
bt_names = bt_parent.get_topic_info().set_index("Topic")["Name"].to_dict()
PT = []
for pt, lis in sorted(ptopic_leaves.items()):
    spans = sorted({shelf_idx[leaf_shelf[li]] for li in lis})
    PT.append({
        "i": int(pt),
        "label": str(bt_names.get(pt, pt)),
        "size": sum(len(leaf_members[li]) for li in lis),
        "spans": spans,
        "leaves": [{"s": shelf_idx[leaf_shelf[li]], "label": leaf_label[li],
                    "n": len(leaf_members[li])} for li in lis],
    })
PT.sort(key=lambda x: (-len(x["spans"]), -x["size"]))   # cross-shelf topics first

# Right pane: child shelves with their leaf themes.
CS = []
for si, lab in enumerate(child_shelf_order):
    themes_here = [{"label": leaf_label[li], "n": len(leaf_members[li])}
                   for li in range(n_leaf) if leaf_shelf[li] == lab]
    CS.append({"i": si, "label": lab, "n_themes": len(themes_here),
               "n_chunks": sum(t["n"] for t in themes_here), "themes": themes_here})

CSS = '''<style>
:root{--brand:#d53355;--brandg:#646d1a;--header:#173f35;--panel:#f6efe6;--line:#e8e1d4;
--muted:#7a6657;--fg:#1f1a15;--sel:#f7d6dd;--hl:#edf0d5;}
*{box-sizing:border-box}body{margin:0;font:14px/1.45 "Quicksand",system-ui,sans-serif;color:var(--fg)}
header{background:var(--header);color:#f6efe6;padding:10px 16px;display:flex;gap:12px;align-items:center}
header button{margin-left:auto;background:#f6efe6;border:0;border-radius:8px;padding:4px 12px;cursor:pointer}
.cols{display:flex;height:calc(100vh - 80px)}
.col{flex:1;overflow:auto;padding:12px 14px}.col.l{border-right:1px solid var(--line)}
h3{margin:0 0 10px;font-size:13px;color:var(--muted);text-transform:uppercase;letter-spacing:.4px}
.card{background:var(--panel);border-radius:10px;margin:7px 0;padding:8px 10px;border-left:3px solid var(--brand);cursor:pointer}
.col.r .card{border-left-color:var(--brandg)}
.card.on{background:var(--sel)}.card.hl{outline:2px solid var(--brandg);background:#fff}
.t{font-weight:600}.meta{color:var(--muted);font-size:12px;margin-left:6px}
.sub{font-size:12px;margin-top:5px}
.chip{display:inline-block;background:#fff;border:1px solid var(--line);border-radius:999px;
padding:1px 8px;margin:2px 4px 0 0;font-size:11.5px}
.chip.hl{background:var(--hl);border-color:var(--brandg)}
.tt{font-size:12.5px;padding:3px 0;border-top:1px solid var(--line)}
.tt.hl{background:var(--hl);border-radius:6px;padding:3px 6px}
.span{color:var(--brand);font-weight:700}
</style>'''

JS = ("<script>\nconst PT=" + _json.dumps(PT) + ";\nconst CS=" + _json.dumps(CS) + r''';
let selP=null, selS=null;
const L=document.getElementById("L"), R=document.getElementById("R"), H=document.getElementById("h");
function esc(s){return String(s==null?"":s).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");}
function render(){
  L.innerHTML = PT.filter(p => selS===null || p.spans.includes(selS)).map(p=>{
    const on = selP===p.i ? " on":"";
    const chips = p.leaves.map(lt=>{
      const hl = (selS!==null && lt.s===selS) ? " hl":"";
      return `<span class="chip${hl}">${esc(lt.label)} · ${lt.n}</span>`;}).join("");
    const spanTxt = p.spans.length>1 ? `<span class="span"> · spans ${p.spans.length} shelves</span>`:"";
    return `<div class="card${on}" onclick="pickP(${p.i})"><span class="t">${esc(p.label)}</span>`+
      `<span class="meta">${p.size} chunks · ${p.leaves.length} leaf themes${spanTxt}</span>`+
      `<div class="sub">${chips}</div></div>`;}).join("") || "<p class='meta'>none for this shelf</p>";
  const fed = selP!==null ? new Set(PT.find(p=>p.i===selP).spans) : null;
  const fedT = selP!==null ? new Set(PT.find(p=>p.i===selP).leaves.map(c=>c.label)) : null;
  R.innerHTML = CS.map(s=>{
    const on = selS===s.i ? " on":"";
    const hl = (fed && fed.has(s.i)) ? " hl":"";
    const tts = s.themes.map(t=>{
      const th = (fedT && fedT.has(t.label)) ? " hl":"";
      return `<div class="tt${th}">${esc(t.label)} <span class="meta">${t.n}</span></div>`;}).join("");
    return `<div class="card${on}${hl}" onclick="pickS(${s.i})"><span class="t">${esc(s.label)}</span>`+
      `<span class="meta">${s.n_themes} themes · ${s.n_chunks} chunks</span>${tts}</div>`;}).join("");
  H.textContent = `${PT.length} parent topics · ${CS.length} child shelves`+
    (selP!==null?` · topic: ${PT.find(p=>p.i===selP).label}`:"")+
    (selS!==null?` · shelf: ${CS[selS].label}`:"");
}
function pickP(i){selP=(selP===i?null:i);render();}
function pickS(i){selS=(selS===i?null:i);render();}
function clearSel(){selP=null;selS=null;render();}
render();
</script>''')

BODY = (f"<header><b>{_hh.escape(P.label)}</b> — BERTopic theme hierarchy ↔ child shelves "
        f"<span id='h' class='meta'></span><button onclick=\"clearSel()\">clear</button></header>"
        "<div class='cols'>"
        "<div class='col l'><h3>BERTopic parent topics (rollup of leaf themes)</h3><div id='L'></div></div>"
        "<div class='col r'><h3>Child shelves &amp; their leaf themes</h3><div id='R'></div></div>"
        "</div>")
page = ("<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'><title>" +
        _hh.escape(P.label) + " — bertopic rollup</title>" + CSS + "</head><body>" + BODY + JS + "</body></html>")

VIZ = ROOT / "data" / "viz"; VIZ.mkdir(parents=True, exist_ok=True)
fn = VIZ / f"bertopic_rollup_{P.shelf_id.replace(':', '_').replace('/', '_')}.html"
fn.write_text(page, encoding="utf-8")
print("wrote", fn, f"({len(PT)} parent topics, {len(CS)} child shelves)")
IFrame(src=f"../data/viz/{fn.name}", width="100%", height=720)


## 14 · Collapsible theme dendrogram

The same BERTopic hierarchy as §12's ASCII `get_topic_tree`, but as an **interactive collapsible tree**:
merge nodes → flat parent-topics → leaf themes (tagged by child shelf, with chunk counts). Click any
branch to expand/collapse; **expand-all / collapse-all** buttons in the header. Parent-topics that
**span >1 child shelf** are flagged — the cross-shelf groupings BERTopic found. Reuses `hier`,
`parent_topics`, and the `leaf_*` arrays from §12; saved to `data/viz/`.


In [ ]:
import html as _h4
from IPython.display import IFrame

# --- 1. parse hierarchical_topics() into a node graph -------------------------------------------
# Columns (confirmed): Parent_ID, Parent_Name, Topics, Child_Left_ID, Child_Left_Name,
# Child_Right_ID, Child_Right_Name, Distance. IDs come back as str; coerce to int so they unify
# with the flat parent-topic ids (0..K-1).
def _i(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        return x

node_name, children = {}, {}           # id -> display name ; id -> [child ids]
child_ids = set()
for _, r in hier.iterrows():
    pid = _i(r["Parent_ID"])
    lid, rid = _i(r["Child_Left_ID"]), _i(r["Child_Right_ID"])
    node_name[pid] = str(r["Parent_Name"])
    node_name.setdefault(lid, str(r["Child_Left_Name"]))
    node_name.setdefault(rid, str(r["Child_Right_Name"]))
    children[pid] = [lid, rid]
    child_ids.update([lid, rid])
roots = [nid for nid in node_name if nid not in child_ids]

# flat parent-topic id -> the leaf themes it grouped (from §12)
flat_leaves = {}
for li, ptid in enumerate(parent_topics):
    flat_leaves.setdefault(int(ptid), []).append(li)
flat_ids = set(flat_leaves)            # these are the dendrogram leaves


def esc(s):
    return _h4.escape(str(s))


def render_node(nid, depth=0):
    kids = children.get(nid)
    # a dendrogram leaf == a flat parent-topic: expand to its leaf themes
    if nid in flat_ids and not kids:
        lis = flat_leaves.get(nid, [])
        shelves = sorted({leaf_shelf[i] for i in lis})
        span = f" <span class='span'>· spans {len(shelves)} shelves</span>" if len(shelves) > 1 else ""
        body = "".join(
            f"<div class='leaf'><span class='dot'></span>{esc(leaf_label[i])}"
            f"<span class='n'>{len(leaf_members[i])}</span></div>" for i in lis)
        return (f"<details open><summary class='topic'>{esc(node_name.get(nid, nid))}"
                f"<span class='n'>{len(lis)} themes</span>{span}</summary>{body}</details>")
    if not kids:                        # bare leaf (rare)
        return f"<div class='leaf'>{esc(node_name.get(nid, nid))}</div>"
    inner = "".join(render_node(c, depth + 1) for c in kids)
    op = " open" if depth < 1 else ""   # top levels open, deeper ones collapsed by default
    return (f"<details{op}><summary class='merge'>{esc(node_name.get(nid, nid))}</summary>"
            f"<div class='kids'>{inner}</div></details>")

tree_html = "".join(render_node(r) for r in roots)

CSS = '''<style>
:root{--brand:#d53355;--brandg:#646d1a;--header:#173f35;--panel:#f6efe6;--line:#e8e1d4;--muted:#7a6657;--fg:#1f1a15;}
*{box-sizing:border-box}body{margin:0;font:14px/1.5 "Quicksand",system-ui,sans-serif;color:var(--fg)}
header{background:var(--header);color:#f6efe6;padding:10px 16px;display:flex;gap:12px;align-items:center}
header button{background:#f6efe6;border:0;border-radius:8px;padding:4px 12px;cursor:pointer;font:inherit}
.wrap{padding:14px 18px;max-width:1100px}
details{margin:2px 0}.kids{margin-left:18px;border-left:1px dashed var(--line);padding-left:12px}
summary{cursor:pointer;padding:3px 6px;border-radius:6px;list-style:none}
summary::-webkit-details-marker{display:none}
summary::before{content:"▸";display:inline-block;width:1em;color:var(--muted);transition:transform .1s}
details[open]>summary::before{transform:rotate(90deg)}
summary.merge{color:var(--muted);font-style:italic}
summary.topic{font-weight:700;background:var(--panel);border-left:3px solid var(--brand)}
.leaf{margin:2px 0 2px 26px;font-size:12.5px}
.dot{display:inline-block;width:6px;height:6px;border-radius:50%;background:var(--brandg);margin-right:7px;vertical-align:middle}
.n{color:var(--brandg);font-size:11.5px;margin-left:8px;font-variant-numeric:tabular-nums}
.span{color:var(--brand);font-weight:700;font-size:11.5px}
</style>'''
JS = '''<script>
function setAll(open){document.querySelectorAll("details").forEach(d=>d.open=open);}
</script>'''
page = ("<!DOCTYPE html><html><head><meta charset='utf-8'>" + CSS + "</head><body>" +
        f"<header><b>{esc(P.label)}</b> — BERTopic theme dendrogram (collapsible)"
        "<button onclick='setAll(true)'>expand all</button>"
        "<button onclick='setAll(false)'>collapse all</button></header>"
        f"<div class='wrap'>{tree_html}</div>" + JS + "</body></html>")

VIZ = ROOT / "data" / "viz"; VIZ.mkdir(parents=True, exist_ok=True)
fn = VIZ / f"bertopic_tree_{P.shelf_id.replace(':', '_').replace('/', '_')}.html"
fn.write_text(page, encoding="utf-8")
print("wrote", fn, f"({len(roots)} root(s), {len(flat_ids)} parent topics, {len(leaf_label)} leaf themes)")
IFrame(src=f"../data/viz/{fn.name}", width="100%", height=720)


## 15 · Per-shelf BERTopic themes — two-pane explorer (like `layer_a_tree_foods.html`)

For **every shelf** we build a BERTopic theme over **all of that shelf's descendant documents**, then
bake it into the **same two-pane HTML app as `data/viz/layer_a_tree_foods.html`**: a collapsible shelf
**tree on the left**, a per-shelf **detail pane on the right**.

- **Left pane** — the shelf hierarchy (every top-level foods shelf and its subtree). Click a caret to
  expand/collapse; each row shows the node's subtree-doc count and a `[N]` topic badge.
- **Right pane** — click a shelf to see its **theme label** (one concise title, ≤6 words, written by
  the §1 LLM provider; keyword fallback when no key) and its **flat topics** over the subtree docs.
  **Each topic is LLM-named** too: one *batch* call per node turns that node's c-TF-IDF keyword-topics
  into short human labels (the keywords stay visible as a sub-line). Each topic expands to a few
  **representative docs**, badged **direct** (the doc is attached to this shelf itself) or **indirect**
  (it came from a descendant) — the same direct/indirect semantics the Layer A tree uses.
- **Scope per node = the node's full subtree** (own docs + every descendant's). **K is auto-by-size**:
  `K = clamp(round(sqrt(n/2)), 2, 12)`, KMeans on raw BGE vectors (the §4 recipe), full coverage. Nodes
  with too few cached-vector docs (`< MIN_CHUNKS`) appear greyed with no fit.
- Self-contained HTML (data baked as JSON, vanilla JS, WiseFood palette). Saved to
  `data/viz/bertopic_pernode_foods_facet.html`.

> Same explorer shell as the Layer A tree — but the right pane shows BERTopic's per-shelf themes over
> all-descendant docs instead of Layer B's themes.

In [22]:
import html as _h15
import json as _json15
import math
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from IPython.display import IFrame

# ── roots to walk: every TOP-LEVEL foods shelf (each direct child of the synthetic `facet:foods`
# root) is walked as its own subtree. We build a per-shelf BERTopic theme over the shelf's ALL
# descendant docs, then bake everything into a two-pane HTML app (left tree / right detail) modelled
# on data/viz/layer_a_tree_foods.html.
SYNTH_ROOT_ID = "facet:foods"
_facet_root = fs.graph.shelf(SYNTH_ROOT_ID)
if _facet_root is not None:
    ROOT_SHELVES = _facet_root.children()
else:  # no synthetic root in this build → foods shelves with no parent (or parent outside foods)
    _foods = fs.graph.shelves(facet="foods")
    _ids = {s.shelf_id for s in _foods}
    ROOT_SHELVES = [s for s in _foods
                    if s.parent_shelf_id is None or s.parent_shelf_id not in _ids]
ROOT_SHELVES = sorted(ROOT_SHELVES, key=lambda s: s.chunk_count, reverse=True)
print(f"walking {len(ROOT_SHELVES)} top-level foods shelves: "
      + ", ".join(f"{s.label}" for s in ROOT_SHELVES[:10])
      + (" …" if len(ROOT_SHELVES) > 10 else ""))

MIN_CHUNKS = 6          # below this we don't fit BERTopic (too few points to cluster)
REP_DOCS = 6            # representative docs shown per topic in the detail pane
SNIPPET = 240           # chars of doc text shown per representative doc
_REDUCER15 = BaseDimensionalityReduction()
_VEC15 = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
_CTFIDF15 = ClassTfidfTransformer(reduce_frequent_words=True)

# ── per-node LLM theme label (reuses §1 provider/key/model; keyword fallback when unavailable) ────
_LLM15 = None
_BASE_URL15 = {"openrouter": "https://openrouter.ai/api/v1",
               "groq": "https://api.groq.com/openai/v1", "openai": None}
if LLM_ENABLED and LLM_PROVIDER in _BASE_URL15:
    try:
        import openai
        _kw15 = {"api_key": os.environ[_ENV_VAR]}
        if _BASE_URL15[LLM_PROVIDER]:
            _kw15["base_url"] = _BASE_URL15[LLM_PROVIDER]
        _LLM15 = openai.OpenAI(**_kw15)
        print(f"per-node theme labels via {LLM_PROVIDER}/{LLM_MODEL}")
    except Exception as e:
        print("per-node LLM labels unavailable, using keyword titles:", repr(e)[:120])
else:
    print("per-node theme labels: no OpenAI-compatible LLM → keyword titles from top topics")


def _keyword_title(shelf_label, topics):
    words, seen = [], set()
    for t in topics[:3]:
        for w in str(t["label"]).replace("_", " ").split():
            w = w.strip(",·").lower()
            if w and not w.isdigit() and w not in seen:
                seen.add(w); words.append(w)
            if len(words) >= 4:
                break
        if len(words) >= 4:
            break
    return (", ".join(words) or shelf_label).title()


def _theme_label(shelf_label, topics):
    if not topics:
        return None
    if _LLM15 is None:
        return _keyword_title(shelf_label, topics)
    bullets = "\n".join(f"- {t['label']} ({t['n']} docs)" for t in topics[:12])
    prompt = ("These are the BERTopic topics found in documents under the food category "
              f"'{shelf_label}':\n{bullets}\n\n"
              "Give ONE concise theme label (<=6 words, no quotes, no trailing punctuation) that "
              "summarizes what this category's documents are collectively about.\n"
              "Answer with the label only.")
    try:
        r = _LLM15.chat.completions.create(
            model=LLM_MODEL, temperature=0.0, max_tokens=24,
            messages=[{"role": "user", "content": prompt}])
        txt = (r.choices[0].message.content or "").strip().strip('"').splitlines()[0].strip()
        return txt or _keyword_title(shelf_label, topics)
    except Exception:
        return _keyword_title(shelf_label, topics)


def _name_topics(shelf_label, topic_inputs):
    """ONE LLM call that names ALL of a node's topics together. `topic_inputs` is a list of
    {keywords, sample} dicts; returns a list of short labels (same length/order). Falls back to the
    c-TF-IDF keyword string per topic if the LLM is unavailable or the response can't be parsed."""
    fallback = [ti["keywords"] for ti in topic_inputs]
    if _LLM15 is None or not topic_inputs:
        return fallback
    blocks = "\n".join(
        f'{i}. keywords: {ti["keywords"]}\n   sample: {ti["sample"]}'
        for i, ti in enumerate(topic_inputs))
    prompt = ("You are labeling document topics found under the food category "
              f"'{shelf_label}'. For each numbered topic below, write a concise (<=6 words) human "
              "label — no quotes, no trailing punctuation.\n\n"
              f"{blocks}\n\n"
              'Return ONLY a JSON array of strings, one label per topic in order, e.g. '
              '["Whole-grain oat fiber", "Rice starch and gluten"]. '
              f"The array must have exactly {len(topic_inputs)} items.")
    try:
        r = _LLM15.chat.completions.create(
            model=LLM_MODEL, temperature=0.0, max_tokens=40 + 16 * len(topic_inputs),
            messages=[{"role": "user", "content": prompt}])
        raw = (r.choices[0].message.content or "").strip()
        s, e = raw.find("["), raw.rfind("]")
        labels = _json15.loads(raw[s:e + 1]) if s != -1 and e != -1 else []
        out = []
        for i in range(len(topic_inputs)):
            lab = str(labels[i]).strip().strip('"') if i < len(labels) else ""
            out.append(lab or fallback[i])
        return out
    except Exception:
        return fallback


def _auto_k(n):
    return max(2, min(12, round(math.sqrt(n / 2))))


def _own_chunk_ids(sh):
    """Chunks attached directly to this shelf (DIRECT membership)."""
    return {c.chunk_id for c in sh.chunks()}


def _subtree_chunk_ids(sh):
    """Every chunk in this node's subtree (itself + all descendants), deduped."""
    ids = _own_chunk_ids(sh)
    for kid in sh.children():
        ids |= _subtree_chunk_ids(kid)
    return ids


def _fit_node(sh):
    """Per-shelf BERTopic theme over the shelf's ALL descendant docs. Each topic keeps a few
    representative docs tagged direct (on this shelf) / indirect (from a descendant). Recurses."""
    own = _own_chunk_ids(sh)
    sub = _subtree_chunk_ids(sh)
    cc = [c for c in sub if c in id2vec]                 # only chunks with a cached vector
    node = {"id": sh.shelf_id, "label": sh.label, "depth": getattr(sh, "depth", 0),
            "foodon_id": getattr(sh, "foodon_id", None),
            "n_sub": len(sub), "n_vec": len(cc), "n_direct": len(own & set(cc)),
            "theme": None, "topics": [], "skipped": None, "children": []}

    if len(cc) >= MIN_CHUNKS:
        recs = fs.chunk_store.get_many(list(cc))
        by_id = {c.chunk_id: c for c in recs}
        order = [cid for cid in cc if cid in by_id]
        M = np.asarray([id2vec[cid] for cid in order])
        docs_n = [by_id[cid].text for cid in order]
        K = min(_auto_k(len(order)), len(order) - 1)
        bt = BERTopic(umap_model=_REDUCER15,
                      hdbscan_model=KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10),
                      vectorizer_model=_VEC15, ctfidf_model=_CTFIDF15,
                      calculate_probabilities=False, verbose=False)
        try:
            tn = np.asarray(bt.fit_transform(docs_n, embeddings=M)[0])
            info = bt.get_topic_info().set_index("Topic")
            names, counts = info["Name"].to_dict(), info["Count"].to_dict()
            # build topics with c-TF-IDF keywords first; collect inputs for ONE batch LLM naming call
            name_inputs = []
            for t in sorted(set(int(x) for x in tn) - {-1}):
                idx = [i for i in range(len(order)) if int(tn[i]) == t]
                docs_out = []
                for i in idx[:REP_DOCS]:
                    cid = order[i]; ch = by_id[cid]
                    docs_out.append({
                        "link": "direct" if cid in own else "indirect",
                        "source": ch.source_doc_id or "",
                        "snippet": (ch.text or "")[:SNIPPET],
                    })
                kw_terms = [w for w, _ in (bt.get_topic(t) or [])][:8]
                keywords = ", ".join(kw_terms) or str(names.get(t, t))
                node["topics"].append({
                    "label": keywords,            # placeholder until LLM names it below
                    "keywords": keywords,
                    "n": int(counts.get(t, len(idx))),
                    "n_direct": sum(1 for i in idx if order[i] in own),
                    "docs": docs_out,
                })
                name_inputs.append({"keywords": keywords,
                                    "sample": (docs_out[0]["snippet"] if docs_out else "")[:200]})
            # ONE LLM call names all this node's topics; theme summarizes the *named* topics
            llm_names = _name_topics(sh.label, name_inputs)
            for tp, nm in zip(node["topics"], llm_names):
                tp["label"] = nm
            node["theme"] = _theme_label(sh.label, node["topics"])
        except Exception as e:
            node["skipped"] = f"fit failed: {repr(e)[:120]}"
    else:
        node["skipped"] = f"only {len(cc)} vectored chunks (< {MIN_CHUNKS})"

    node["children"] = [_fit_node(kid) for kid in sh.children()]
    return node


forest = [_fit_node(s) for s in ROOT_SHELVES]   # one fitted tree per top-level shelf

# ── text summary so the cell is useful even without opening the HTML ─────────────────────────────
def _walk(n, d=0):
    tag = (f"“{n['theme']}” · {len(n['topics'])} topics" if n["topics"]
           else f"— skipped ({n['skipped']})")
    print(f"{'  ' * d}{n['label'][:40]:40}  n={n['n_vec']:>4} (dir {n['n_direct']:>3})  {tag}")
    for c in n["children"]:
        _walk(c, d + 1)

print()
for _tree in forest:
    _walk(_tree)

_n_nodes = 0
def _count(n):
    global _n_nodes
    _n_nodes += 1
    for c in n["children"]:
        _count(c)
for _tree in forest:
    _count(_tree)

TREE_DATA15 = {"meta": {"facet": "foods", "n_top": len(forest), "n_nodes": _n_nodes},
               "roots": forest}

# ── two-pane HTML app (left tree / right detail), modelled on layer_a_tree_foods.html ─────────────
_TMPL15 = r"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><title>__TITLE__</title>
<style>
:root{--brand:#d53355;--brandg:#a6b52b;--earth-1:#f6efe6;--bg:#fff;--fg:#1f1a15;--muted:#7a6657;
--line:#e8e1d4;--panel:#f6efe6;--panel-2:#efe7d8;--header:#173f35;--header-fg:#f6efe6;--accent:#646d1a;
--accent-soft:#edf0d5;--sel:#f7d6dd;--chip-bg:#edf0d5;--chip-fg:#424811;--radius:12px;
--font-body:"Quicksand",ui-sans-serif,system-ui,-apple-system,"Segoe UI",sans-serif;
--font-display:ui-rounded,"Quicksand","SF Pro Rounded",system-ui,sans-serif;}
*{box-sizing:border-box}body{margin:0;font:14px/1.45 var(--font-body);color:var(--fg);background:var(--bg)}
header{padding:12px 18px;background:var(--header);color:var(--header-fg);display:flex;align-items:baseline;gap:12px}
header b{font-size:15px;font-family:var(--font-display)}header .counts{opacity:.82;font-size:12.5px}
.wrap{display:flex;height:calc(100vh - 48px)}
.tree{width:44%;overflow:auto;border-right:1px solid var(--line);padding:10px 0;background:var(--panel)}
.detail{flex:1;overflow:auto;padding:18px 22px}
.row{display:flex;align-items:center;gap:7px;padding:3px 10px;cursor:pointer;white-space:nowrap;border-radius:6px;margin:0 6px}
.row:hover{background:var(--accent-soft)}.row.sel{background:var(--sel)}.row.sub{color:var(--muted)}
.caret{width:12px;display:inline-block;color:var(--muted)}
.dot{width:8px;height:8px;border-radius:50%;display:inline-block}
.cc{color:var(--muted);font-variant-numeric:tabular-nums}
.tn{color:var(--accent);font-variant-numeric:tabular-nums;font-weight:600}
.kids{margin-left:16px}.hidden{display:none}
h2{margin:0 0 2px;font-size:19px;font-family:var(--font-display)}
.theme-line{color:var(--header);font-weight:700;font-style:italic;margin:0 0 4px}
.sub-meta{color:var(--muted);margin-bottom:14px}
.origin{margin:14px 0 6px;font-weight:600;color:var(--accent)}
.topic{padding:7px 11px;border-left:3px solid var(--accent);margin:5px 0;background:var(--panel);
border-radius:0 var(--radius) var(--radius) 0}
.topic .kw{color:var(--muted);font-size:12px;margin-top:2px}
.topic details{margin-top:6px}.topic details>summary{cursor:pointer;color:var(--muted);font-size:12px}
.chunk{border-top:1px solid var(--line);padding:5px 2px 4px}
.chunk .src{color:var(--muted);font-size:11px}.chunk .tx{font-size:12.5px;margin-top:2px}
.b{font-size:10px;font-weight:700;border-radius:999px;padding:1px 7px;margin-right:6px}
.b.direct{background:var(--chip-bg);color:var(--chip-fg)}
.b.indirect{background:#f3e0d6;color:#7a3f24}
.empty{color:var(--muted);font-style:italic}
</style></head><body>
<header><b id="h-title"></b><span class="counts" id="h-counts"></span></header>
<div class="wrap"><div class="tree" id="tree"></div>
<div class="detail" id="detail"><p class="empty">Select a shelf on the left.</p></div></div>
<script>
const TREE_DATA = __TREE_JSON__;
function esc(s){return String(s==null?"":s).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");}
function nTopics(n){return (n.topics||[]).length;}
function renderTree(){
  const m=TREE_DATA.meta||{};
  document.getElementById("h-title").textContent="BERTopic per-shelf themes — "+(m.facet||"");
  document.getElementById("h-counts").textContent=(m.n_top||0)+" top-level shelves · "+(m.n_nodes||0)+" nodes";
  const root=document.getElementById("tree");root.innerHTML="";
  TREE_DATA.roots.forEach(n=>root.appendChild(rowFor(n,true)));
}
function rowFor(node,open){
  const box=document.createElement("div");const row=document.createElement("div");
  const fit=nTopics(node)>0;row.className="row"+(fit?"":" sub");
  const hasKids=node.children&&node.children.length;const tc=nTopics(node);
  const caret=hasKids?(open?"▾":"▸"):" ";
  row.innerHTML='<span class="caret">'+caret+'</span>'+
    '<span class="dot" style="background:'+(fit?"var(--accent)":"var(--line)")+'"></span>'+
    '<span class="lbl">'+esc(node.label)+'</span>'+
    '<span class="cc">'+node.n_vec+'</span>'+
    (tc?'<span class="tn">['+tc+']</span>':'');
  let kids=null;
  if(hasKids){kids=document.createElement("div");kids.className="kids"+(open?"":" hidden");
    node.children.forEach(c=>kids.appendChild(rowFor(c,false)));}
  row.onclick=(ev)=>{ev.stopPropagation();
    if(hasKids&&ev.target.classList.contains("caret")){
      kids.classList.toggle("hidden");
      ev.target.textContent=kids.classList.contains("hidden")?"▸":"▾";return;}
    document.querySelectorAll(".row.sel").forEach(r=>r.classList.remove("sel"));
    row.classList.add("sel");showDetail(node);};
  box.appendChild(row);if(kids)box.appendChild(kids);return box;
}
function showDetail(node){
  const d=document.getElementById("detail");
  let html="<h2>"+esc(node.label)+"</h2>";
  if(node.theme)html+='<div class="theme-line">“'+esc(node.theme)+'”</div>';
  html+='<div class="sub-meta">'+(node.foodon_id?esc(node.foodon_id)+" · ":"")+
    "depth "+node.depth+" · "+node.n_vec+" subtree docs ("+node.n_direct+" direct) · "+
    nTopics(node)+" topics</div>";
  if(!nTopics(node)){html+="<p class='empty'>"+esc(node.skipped||"no topics")+"</p>";
    d.innerHTML=html;return;}
  html+='<div class="origin">FLAT TOPICS ('+nTopics(node)+') over all-descendant docs</div>';
  node.topics.forEach(t=>{
    const ch=t.docs||[];const ni=(t.n||0)-(t.n_direct||0);
    let block='<div class="topic">'+esc(t.label)+' <span class="cc">'+t.n+'doc</span>'+
      (t.keywords&&t.keywords!==t.label?'<div class="kw">'+esc(t.keywords)+'</div>':'');
    if(ch.length){
      block+="<details><summary>"+t.n+" docs · "+(t.n_direct||0)+" direct / "+ni+
        " indirect — showing "+ch.length+"</summary>";
      ch.forEach(c=>{block+='<div class="chunk"><span class="b '+c.link+'">'+c.link+"</span>"+
        '<span class="src">'+esc(c.source)+"</span>"+
        '<div class="tx">'+esc(c.snippet)+"</div></div>";});
      block+="</details>";}
    html+=block+"</div>";});
  d.innerHTML=html;
}
renderTree();
</script></body></html>"""

page15 = (_TMPL15
          .replace("__TITLE__", "BERTopic per-shelf themes — foods")
          .replace("__TREE_JSON__", _json15.dumps(TREE_DATA15)))

VIZ = ROOT / "data" / "viz"; VIZ.mkdir(parents=True, exist_ok=True)
fn15 = VIZ / "bertopic_pernode_foods_facet.html"
fn15.write_text(page15, encoding="utf-8")
print("\nwrote", fn15, f"({_n_nodes} nodes)")
IFrame(src=f"../data/viz/{fn15.name}", width="100%", height=760)


walking 6 top-level foods shelves: plant food product, animal food product, multi-component food, condiment, food material analog, cultural food
per-node theme labels via groq/llama-3.1-8b-instant


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completion


plant food product                        n=2881 (dir 184)  “Nutrition and Food Science Basics” · 12 topics
  botanical fruit food product              n=  67 (dir   3)  “Fruit and overall health benefits” · 6 topics
    legume food product                       n=  64 (dir  10)  “Legume food nutrition and preparation” · 6 topics
      pulse food product                        n=  54 (dir   3)  “Healthy Legume Based Eating” · 5 topics
        dry bean pod                              n=  25 (dir  25)  “Dry bean nutritional benefits” · 4 topics
        hummus                                    n=  27 (dir  27)  “Legume based vegetarian eating habits” · 4 topics
  cruciferous food product                  n=   8 (dir   8)  “Cancer prevention through cruciferous foods” · 2 topics
  maize (corn) food product                 n=  94 (dir  19)  “Maize food product research and analysis” · 7 topics
    corn kernel                               n=  83 (dir  83)  “Corn Kernel Nutrition and Uses